In [1]:
!pip install -q trl peft bitsandbytes transformers accelerate datasets huggingface_hub evaluate sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.3/366.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 26.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/20

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOTrainer, DPOConfig
from trl.trainer.utils import DPODataCollatorWithPadding
from datasets import load_dataset
from huggingface_hub import login
import torch


# ✅ 1. Hugging Face Login
HF_TOKEN = " "  # Replace with your token
login(token=HF_TOKEN)


2025-06-12 14:00:42.704360: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749736842.914375      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749736842.975307      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
# ✅ 2. Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# ✅ 3. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ✅ 4. Load base model and attach SFT adapter
sft_model = AutoModelForCausalLM.from_pretrained(
    "TK47/tinyllama-sft-t3",
    torch_dtype=torch.bfloat16
).to(device)


# sft_model = PeftModel.from_pretrained(
#     base_model,
#     model_id= "TK47/tinyllama-sft-trials",
#     subfolder= "trial_3",
# )

cuda


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/18.0M [00:00<?, ?B/s]

In [4]:
# ✅ 5. Apply new LoRA for DPO
dpo_lora_config = LoraConfig(
    r=8,
    lora_alpha=128,
    lora_dropout=0.2,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(sft_model, dpo_lora_config)

model.print_trainable_parameters()

   # TRIAL_CONFIGS = [
   #      # Trial 1
   #      {was with 500 rows and 200 eval
   #          "lora": {
   #              "r": 8,
   #              "alpha": 128,
   #              "target_modules": ["q_proj", "v_proj"],
   #              "dropout": 0.2
   #          },
   #          "dpo": {
   #              "beta": 0.1,
   #              "lr": 5e-4,
   #              "batch_size":4,
   #              "epochs": 3
#gradient_accumulation_steps=2
   #          }
   #      },
   #      # Trial 2
   #      { was with 500 rows and 250 eval
   #          "lora": {
   #              "r": 8,
   #              "alpha": 32,
   #              "target_modules": ["q_proj", "v_proj", "k_proj"],
   #              "dropout": 0.1
   #          },
   #          "dpo": {
   #              "beta": 0.2,
   #              "lr": 5e-4,
   #              "batch_size": 3,
   #              "epochs": 2
#gradient_accumulation_steps=2
   #          }
   #      },
   #      # Trial 3
   #      { t3 - t4 1000 test rows 500 eval
   #          "lora": {
   #              "r": 16,
   #              "alpha": 64,
   #              "target_modules": ["q_proj"],
   #              "dropout": 0.2
   #          },
   #          "dpo": {
   #              "beta": 0.05,
   #              "lr": 2e-4,
   #              "batch_size": 4,
   #              "epochs": 3
#gradient_accumulation_steps=2
   #          }
   #      },
   #      # Trial 4
   #      {
   #          "lora": {
   #              "r": 16,
   #              "alpha": 32,
   #              "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
   #              "dropout": 0.1
   #          },
   #          "dpo": {
   #              "beta": 0.15,
   #              "lr": 4e-6,
   #              "batch_size": 2,
   #              "epochs": 1
   #gradient_accumulation_steps=4
   #          }
   #      },
   #      # Trial 5
   #      {
   #          "lora": {
   #              "r": 12,
   #              "alpha": 128,
   #              "target_modules": ["q_proj", "k_proj"],
   #              "dropout": 0.2
   #          },
   #          "dpo": {
   #              "beta": 0.2,
   #              "lr": 5e-6,
   #              "batch_size": 6,
   #              "epochs": 2
#gradient_accumulation_steps=2
   #          }
   #      }
   #  ]



trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [5]:
# Load the Anthropic HH dataset (or whichever you're using)
dataset = load_dataset("argilla/ultrafeedback-binarized-preferences") #, split="train[:10000]")  # subset for safety
dataset_train = dataset["train"].shuffle(seed=42).select(range(500))
dataset_test = dataset["train"].shuffle(seed=42).select(range(200))

#Just rename fields if needed — no extra formatting required
def format_dpo(example):
    return {
        "prompt": example["instruction"],  
        "chosen": example["chosen_response"],
        "rejected": example["rejected_response"]
    }

# Apply formatting and subset for compute safety
ds_train = dataset_train.map(format_dpo)
ds_test = dataset_test.map(format_dpo)
# dataset = dataset.map(format_dpo)



# ✅ Load and pre-tokenize DPO-compatible dataset
# dataset = load_dataset("Anthropic/hh-rlhf") #, split="train[:10000]")  # subset for safety
# dataset_train = dataset["train"].shuffle(seed=42).select(range(10000))
# dataset_test = dataset["test"]


# def tokenize_dpo(example):
#     return {
#         "chosen": tokenizer(example["chosen"], truncation=True, return_tensors="pt")["input_ids"][0].tolist(),
#         "rejected": tokenizer(example["rejected"], truncation=True, return_tensors="pt")["input_ids"][0].tolist(),
#     }

# tokenized_dataset_train = dataset_train.map(tokenize_dpo)
# tokenized_dataset_test =  dataset_train.map(tokenize_dpo)


README.md:   0%|          | 0.00/8.62k [00:00<?, ?B/s]

(…)-00000-of-00001-9dffc9d46d32c335.parquet:   0%|          | 0.00/110M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/63619 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [8]:
# ✅ 7. DPO Training Configuration
dpo_config = DPOConfig(
    beta=0.1,
    learning_rate=5e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    warmup_steps=10,
    bf16=False,
    fp16=True,  
    logging_steps=10,
    eval_strategy="steps",
    save_steps=50,
    #save_total_limit=2,
    output_dir="./tinyllama-sft-dpo-trial1",
    report_to="none",
    push_to_hub=True,
    hub_model_id="TK47/tinyllama-sft-dpo-t3",
    hub_strategy="end",
    hub_private_repo=False
)



# ✅ 8. DPO Trainer Setup
trainer = DPOTrainer(
    model=model,
    ref_model=None,  # uses a frozen copy of current model internally
    args=dpo_config,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    peft_config=dpo_lora_config,
    processing_class=tokenizer 
)


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
# ✅ 9. Train
trainer.train()

# ✅ 10. Push Final Checkpoint
trainer.push_to_hub("dpo_trial_1")

/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
10,0.684000,0.567813,0.140823,-1.087578,0.620000,1.227922,-395.760010,-333.119995,-3.263125,-3.256875
20,0.732700,0.377813,0.841506,-2.259141,0.795000,3.100122,-389.079987,-345.000000,-3.205625,-3.203125
30,0.800200,0.160771,1.488281,-2.511406,0.970000,4.002500,-382.480011,-347.519989,-3.224375,-3.220000
40,0.202400,0.096348,1.782109,-3.234062,0.985000,5.018750,-379.519989,-354.839996,-3.213750,-3.211250
50,0.108200,0.055843,1.322266,-5.279375,1.000000,6.603750,-384.079987,-375.239990,-3.123750,-3.118438
60,0.112200,0.076666,0.119004,-7.827500,0.980000,7.938750,-396.200012,-400.359985,-3.081875,-3.072813
70,0.096200,0.019382,1.591480,-6.050000,1.000000,7.641250,-381.279999,-382.760010,-3.140625,-3.132188


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


In [ ]:
# 12. Save the Model
model.save_pretrained("./tinyllama-lora-sft-dpo-t1")
tokenizer.save_pretrained("./tinyllama-lora-sft-dpo-t1")